# RSNA Knee — Preprocessing (siap pakai untuk notebook modeling lain)

Menghasilkan 3 output yang bisa di-attach sebagai Kaggle Dataset ke notebook lain:

1. **`image_cache/<StudyUID>/<SeriesUID>.npz`** — slice terpilih (16/series) sudah di-resample skala fisik, di-clip per grup sekuens, resize seragam, siap `np.load()` langsung
2. **`series_manifest.csv`** — 1 baris/series: metadata sekuens, `Laterality` (baru diperiksa), path cache, status QC
3. **`study_manifest.csv`** — 1 baris/studi: label (kalau ada), teks laporan yang sudah dibersihkan, bahasa
4. **`group_norm_stats.csv`** — statistik normalisasi per grup sekuens (dipakai konsisten oleh notebook manapun yang baca cache ini)

Urutan pipeline persis mengikuti spesifikasi di bagian 07 dokumen strategi (resample fisik sebelum resize, clip per-grup, dedup, dst).

## 0. Setup

In [ ]:
import os, sys, json, hashlib, traceback, warnings, random, time, re
from pathlib import Path
from collections import Counter, defaultdict
from multiprocessing import Pool

import numpy as np
import pandas as pd
from scipy import ndimage as ndi

warnings.filterwarnings("ignore")
pd.set_option("display.max_colwidth", 160)
RNG = np.random.default_rng(42)
random.seed(42)

T0 = time.time()
def tlog(msg):
    print(f"[t={time.time()-T0:7.1f}s] {msg}", flush=True)

def ensure(pkg, import_name=None):
    import importlib
    name = import_name or pkg
    try:
        importlib.import_module(name); return True
    except ImportError:
        os.system(f"{sys.executable} -m pip install -q {pkg}")
        try:
            importlib.import_module(name); return True
        except Exception as e:
            print(f"[warn] gagal import/install {pkg}: {e}"); return False

HAVE_PYDICOM = ensure("pydicom")
HAVE_LANGDETECT = ensure("langdetect")
if HAVE_PYDICOM:
    import pydicom
if HAVE_LANGDETECT:
    from langdetect import detect, DetectorFactory
    DetectorFactory.seed = 42

OUT_DIR = Path("prep_outputs")
CACHE_DIR = OUT_DIR / "image_cache"
CACHE_DIR.mkdir(parents=True, exist_ok=True)


In [ ]:
kaggle_input = Path("/kaggle/input")
CANDIDATE_ROOTS = [Path("/kaggle/input/rsna-knee-abnormality-detection"),
                    Path("/kaggle/input/competitions/rsna-knee-abnormality-detection"),
                    Path("../data"), Path("data")]
if kaggle_input.exists():
    for pattern in ("*/train.csv", "*/test.csv", "*/*/train.csv", "*/*/test.csv"):
        for hit in kaggle_input.glob(pattern):
            CANDIDATE_ROOTS.insert(0, hit.parent)
DATA_DIR = next((p for p in CANDIDATE_ROOTS if p.exists() and ((p/"train.csv").exists() or (p/"test.csv").exists())), None) \
           or next((p for p in CANDIDATE_ROOTS if p.exists()), CANDIDATE_ROOTS[0])
tlog(f"DATA_DIR: {DATA_DIR} | exists: {DATA_DIR.exists()}")

train = pd.read_csv(DATA_DIR / "train.csv")
train_series = pd.read_csv(DATA_DIR / "train_series.csv")
train = train.drop_duplicates(subset="StudyInstanceUID").reset_index(drop=True)
train_series = train_series.drop_duplicates(subset=["StudyInstanceUID", "SeriesInstanceUID"]).reset_index(drop=True)
tlog(f"train: {train.shape} | train_series: {train_series.shape}")

LABEL_COLS = ["ACL","MCL","Medial Meniscus","Lateral Meniscus","Medial OA","Lateral OA",
              "PF OA","Effusion","Synovitis","Baker's","Contusion","Fracture"]
LABEL_COLS = [c for c in LABEL_COLS if c in train.columns]
gold_studies = set(train.loc[train[LABEL_COLS].notna().any(axis=1), "StudyInstanceUID"])
tlog(f"Studi gold-label: {len(gold_studies)}")

TRAIN_DIR = DATA_DIR / "train_series"

def list_dcm(study, series):
    d = TRAIN_DIR / study / series
    return sorted(d.glob("*.dcm")) if d.exists() else []


## 1. Investigasi tag `Laterality` (belum pernah dicek sebelumnya)

Krusial untuk keputusan augmentasi flip horizontal (bisa menukar makna Medial&harr;Lateral kalau tidak diperhitungkan).

In [ ]:
LAT_SAMPLE_N = min(2000, len(train_series))
lat_sample = train_series.sample(LAT_SAMPLE_N, random_state=17)
lat_values = Counter()
lat_fail = 0

for _, row in lat_sample.iterrows():
    files = list_dcm(row["StudyInstanceUID"], row["SeriesInstanceUID"])
    if not files:
        lat_fail += 1
        continue
    try:
        ds = pydicom.dcmread(files[0], force=True, stop_before_pixels=True)
        lat = getattr(ds, "Laterality", None)
        lat_img = getattr(ds, "ImageLaterality", None)
        lat_values[(str(lat), str(lat_img))] += 1
    except Exception:
        lat_fail += 1

tlog(f"Sampel diuji: {LAT_SAMPLE_N} series (gagal: {lat_fail})")
print("Distribusi (Laterality, ImageLaterality):")
for k, v in lat_values.most_common():
    print(f"  {k}: {v}")

LATERALITY_AVAILABLE = any(k[0] not in ("None", "") or k[1] not in ("None", "") for k in lat_values)
print(f"\n>>> Tag laterality {'TERSEDIA' if LATERALITY_AVAILABLE else 'TIDAK TERSEDIA'} di data ini.")
print(">>> Kalau TIDAK TERSEDIA: JANGAN pakai horizontal flip augmentation sampai ada cara lain memverifikasi sisi lutut,")
print(">>> karena data kemungkinan campur kiri/kanan dan flip bisa menukar makna Medial<->Lateral secara diam-diam.")


## 2. Statistik normalisasi per grup sekuens

Dihitung dari sampel besar, dipakai konsisten oleh proses caching di bawah maupun notebook lain yang membaca cache ini.

In [ ]:
STATS_SAMPLE_N = min(3000, len(train_series))
stats_sample = train_series.sample(STATS_SAMPLE_N, random_state=29)
stats_rows = []

for _, row in stats_sample.iterrows():
    files = list_dcm(row["StudyInstanceUID"], row["SeriesInstanceUID"])
    if not files:
        continue
    try:
        ds = pydicom.dcmread(files[len(files) // 2], force=True)
        arr = ds.pixel_array.astype(np.float64)
        slope = float(getattr(ds, "RescaleSlope", 1.0) or 1.0)
        intercept = float(getattr(ds, "RescaleIntercept", 0.0) or 0.0)
        arr = arr * slope + intercept
        group = f"{row['Anatomical_Plane']}|FS{row['Fluid_Sensitive']}|Fat{row['Fat_Suppression']}"
        stats_rows.append({"group": group, "p1": np.percentile(arr, 1), "p50": np.percentile(arr, 50),
                            "p99": np.percentile(arr, 99), "mean": arr.mean(), "std": arr.std()})
    except Exception:
        continue

stats_df = pd.DataFrame(stats_rows)
group_norm_stats = stats_df.groupby("group").agg(
    p1=("p1", "median"), p99=("p99", "median"), mean=("mean", "mean"), std=("std", "mean"),
    n_sampled=("group", "size")).reset_index()
group_norm_stats.to_csv(OUT_DIR / "group_norm_stats.csv", index=False)
tlog(f"Statistik grup dihitung dari {len(stats_df)} series sampel, {len(group_norm_stats)} grup unik.")
display(group_norm_stats)

GROUP_STATS = {r["group"]: (r["p1"], r["p99"]) for _, r in group_norm_stats.iterrows()}
FALLBACK_STATS = (float(stats_df["p1"].median()), float(stats_df["p99"].median()))


## 3. Pembersihan teks laporan (per studi)

In [ ]:
def clean_report(text):
    if not isinstance(text, str):
        return None
    t = re.sub(r"\[[A-Z_]+\]", " ", text)   # buang placeholder de-identifikasi mis. [DATE]
    t = re.sub(r"\s+", " ", t).strip()
    return t if t else None

def detect_lang_safe(text):
    if not isinstance(text, str) or len(text.strip()) < 8 or not HAVE_LANGDETECT:
        return None
    try:
        return detect(text)
    except Exception:
        return None

train["Report_clean"] = train["Report"].apply(clean_report)
train["language"] = train["Report_clean"].apply(detect_lang_safe)
train["has_gold_label"] = train["StudyInstanceUID"].isin(gold_studies)

study_manifest_cols = ["StudyInstanceUID", "Report_clean", "language", "has_gold_label"] + LABEL_COLS
study_manifest = train[study_manifest_cols].copy()
study_manifest.to_csv(OUT_DIR / "study_manifest.csv", index=False)
tlog(f"study_manifest.csv: {study_manifest.shape}")
print("Distribusi bahasa:")
print(study_manifest["language"].value_counts().head(12))


## 4. Fungsi worker: preprocessing 1 series penuh

In [ ]:
N_SLICES = 16
IMG_SIZE = 224
TARGET_MM_PER_PX = 0.5  # skala fisik seragam sebelum resize piksel

def quick_hash(arr_uint8):
    return hashlib.md5(arr_uint8.tobytes()).hexdigest()

def preprocess_slice(ds, group_key):
    arr = ds.pixel_array.astype(np.float64)
    slope = float(getattr(ds, "RescaleSlope", 1.0) or 1.0)
    intercept = float(getattr(ds, "RescaleIntercept", 0.0) or 0.0)
    arr = arr * slope + intercept

    if str(getattr(ds, "PhotometricInterpretation", "")) == "MONOCHROME1":
        arr = arr.max() - arr

    p1, p99 = GROUP_STATS.get(group_key, FALLBACK_STATS)
    arr = np.clip(arr, p1, p99)
    rng = (p99 - p1) if (p99 - p1) > 1e-6 else 1.0
    arr = (arr - p1) / rng  # -> [0,1] relatif terhadap grup

    ps = getattr(ds, "PixelSpacing", None)
    mm_per_px = float(ps[0]) if ps else TARGET_MM_PER_PX
    zoom_factor = mm_per_px / TARGET_MM_PER_PX
    if abs(zoom_factor - 1.0) > 0.02:
        arr = ndi.zoom(arr, zoom_factor, order=1)

    h, w = arr.shape[:2]
    if h != IMG_SIZE or w != IMG_SIZE:
        pad_h, pad_w = max(0, IMG_SIZE - h), max(0, IMG_SIZE - w)
        if pad_h or pad_w:
            arr = np.pad(arr, ((pad_h // 2, pad_h - pad_h // 2), (pad_w // 2, pad_w - pad_w // 2)), mode="constant")
        h, w = arr.shape[:2]
        top, left = max(0, (h - IMG_SIZE) // 2), max(0, (w - IMG_SIZE) // 2)
        arr = arr[top: top + IMG_SIZE, left: left + IMG_SIZE]
        if arr.shape != (IMG_SIZE, IMG_SIZE):  # jaga-jaga sisa pembulatan
            arr = np.pad(arr, ((0, IMG_SIZE - arr.shape[0]), (0, IMG_SIZE - arr.shape[1])), mode="constant")

    return np.clip(arr * 255, 0, 255).astype(np.uint8)

def process_one_series(args):
    study, series, plane, fs, fatsup = args
    files = list_dcm(study, series)
    if not files:
        return dict(study=study, series=series, status="MISSING_FOLDER", n_cached=0,
                     laterality=None, n_instances=0, error=None)

    group_key = f"{plane}|FS{fs}|Fat{fatsup}"
    idxs = np.linspace(0, len(files) - 1, min(N_SLICES, len(files))).astype(int)
    idxs = sorted(set(idxs.tolist()))

    slices, hashes = [], set()
    laterality = None
    try:
        for i, fi in enumerate(idxs):
            ds = pydicom.dcmread(files[fi], force=True)
            if i == 0:
                laterality = str(getattr(ds, "Laterality", None) or getattr(ds, "ImageLaterality", None) or "")
            arr8 = preprocess_slice(ds, group_key)
            h = quick_hash(arr8)
            if h in hashes:   # exact-duplicate slice -> lewati, jangan dobel-hitung sinyal sama
                continue
            hashes.add(h)
            slices.append(arr8)

        if not slices:
            return dict(study=study, series=series, status="EMPTY_AFTER_DEDUP", n_cached=0,
                         laterality=laterality, n_instances=len(files), error=None)

        stack = np.stack(slices, axis=0)
        out_dir = CACHE_DIR / study
        out_dir.mkdir(parents=True, exist_ok=True)
        out_path = out_dir / f"{series}.npz"
        np.savez_compressed(out_path, slices=stack)
        return dict(study=study, series=series, status="OK", n_cached=len(slices),
                     laterality=laterality, n_instances=len(files), error=None,
                     cache_path=str(out_path), bytes=out_path.stat().st_size)
    except Exception as e:
        return dict(study=study, series=series, status="ERROR", n_cached=0,
                     laterality=laterality, n_instances=len(files), error=f"{e}\n{traceback.format_exc()}")


## 5. Loop utama: paralel, prioritas studi gold-label, checkpoint resumable, dual safety cutoff

In [ ]:
N_WORKERS = 4
BATCH_SIZE = 150
SAFETY_CUTOFF_SECONDS = 7 * 3600
SAFETY_CUTOFF_BYTES = 16 * 1024**3   # 16 GB -- jaga margin dari kuota /kaggle/working ~21GB
DEBUG_LIMIT_SERIES = None

# resume: cek manifest sebelumnya (dataset ter-attach) kalau ada
prior_manifest_candidates = [Path("/kaggle/working/prior_series_manifest.csv"), Path("prior_series_manifest.csv")]
if kaggle_input.exists():
    for hit in kaggle_input.glob("*/series_manifest.csv"):
        prior_manifest_candidates.insert(0, hit)
prior_manifest_path = next((p for p in prior_manifest_candidates if p.exists()), None)

done_keys = set()
MANIFEST_PATH = OUT_DIR / "series_manifest.csv"
MANIFEST_COLS = ["StudyInstanceUID", "SeriesInstanceUID", "Anatomical_Plane", "Fluid_Sensitive", "Fat_Suppression",
                  "Laterality", "n_instances_total", "n_slices_cached", "status", "cache_path", "bytes"]

running_bytes = 0
if prior_manifest_path is not None:
    tlog(f"Checkpoint ketemu: {prior_manifest_path}")
    prior_df = pd.read_csv(prior_manifest_path)
    done_keys = set(zip(prior_df["StudyInstanceUID"], prior_df["SeriesInstanceUID"]))
    prior_df[MANIFEST_COLS].to_csv(MANIFEST_PATH, index=False)
    running_bytes = int(prior_df["bytes"].fillna(0).sum())
    tlog(f"Series sudah selesai sebelumnya: {len(done_keys)}, {running_bytes/1e9:.2f} GB")
else:
    pd.DataFrame(columns=MANIFEST_COLS).to_csv(MANIFEST_PATH, index=False)

# urutan: semua series studi gold-label dulu, baru sisanya urut StudyInstanceUID (deterministik)
gold_series = train_series[train_series["StudyInstanceUID"].isin(gold_studies)]
rest_series = train_series[~train_series["StudyInstanceUID"].isin(gold_studies)].sort_values(["StudyInstanceUID", "SeriesInstanceUID"])
ordered = pd.concat([gold_series, rest_series]).reset_index(drop=True)

keys = list(zip(ordered["StudyInstanceUID"], ordered["SeriesInstanceUID"]))
todo = ordered[[k not in done_keys for k in keys]].reset_index(drop=True)
if DEBUG_LIMIT_SERIES:
    todo = todo.head(DEBUG_LIMIT_SERIES)

tlog(f"Total series: {len(train_series)} | sudah selesai: {len(done_keys)} | akan diproses run ini (maks): {len(todo)}")
work_items = list(zip(todo["StudyInstanceUID"], todo["SeriesInstanceUID"], todo["Anatomical_Plane"],
                       todo["Fluid_Sensitive"], todo["Fat_Suppression"]))


In [ ]:
error_rows = []
n_done_this_run = 0
hit_cutoff = None

if work_items:
    loop_t0 = time.time()
    with Pool(N_WORKERS) as pool:
        for batch_start in range(0, len(work_items), BATCH_SIZE):
            elapsed = time.time() - T0
            if elapsed > SAFETY_CUTOFF_SECONDS:
                hit_cutoff = "waktu"; break
            if running_bytes > SAFETY_CUTOFF_BYTES:
                hit_cutoff = "ukuran output"; break

            batch = work_items[batch_start: batch_start + BATCH_SIZE]
            results = pool.map(process_one_series, batch)

            rows = []
            for r in results:
                rows.append({"StudyInstanceUID": r["study"], "SeriesInstanceUID": r["series"],
                             "Anatomical_Plane": None, "Fluid_Sensitive": None, "Fat_Suppression": None,
                             "Laterality": r.get("laterality"), "n_instances_total": r.get("n_instances", 0),
                             "n_slices_cached": r.get("n_cached", 0), "status": r["status"],
                             "cache_path": r.get("cache_path"), "bytes": r.get("bytes", 0)})
                running_bytes += r.get("bytes", 0) or 0
                if r.get("error"):
                    error_rows.append({"StudyInstanceUID": r["study"], "SeriesInstanceUID": r["series"], "error": r["error"]})
                n_done_this_run += 1

            batch_df = pd.DataFrame(rows)
            meta = todo[["StudyInstanceUID", "SeriesInstanceUID", "Anatomical_Plane", "Fluid_Sensitive", "Fat_Suppression"]]
            batch_df = batch_df.drop(columns=["Anatomical_Plane", "Fluid_Sensitive", "Fat_Suppression"]).merge(
                meta, on=["StudyInstanceUID", "SeriesInstanceUID"], how="left")
            batch_df[MANIFEST_COLS].to_csv(MANIFEST_PATH, mode="a", header=False, index=False)

            elapsed = time.time() - T0
            loop_elapsed = time.time() - loop_t0
            rate = n_done_this_run / max(1e-6, loop_elapsed)
            n_total_done = len(done_keys) + n_done_this_run
            tlog(f"batch {batch_start//BATCH_SIZE+1}: {n_done_this_run}/{len(work_items)} run ini "
                 f"({n_total_done}/{len(train_series)} total = {n_total_done/len(train_series)*100:.1f}%), "
                 f"{running_bytes/1e9:.2f} GB terpakai, {rate*60:.1f} series/menit")
else:
    tlog("Tidak ada series tersisa.")

if error_rows:
    pd.DataFrame(error_rows).to_csv(OUT_DIR / "preprocessing_errors.csv", mode="a",
                                     header=not (OUT_DIR / "preprocessing_errors.csv").exists(), index=False)

n_total_done = len(done_keys) + n_done_this_run
is_complete = (n_total_done >= len(train_series)) and hit_cutoff is None
print(f"\n{'='*60}\nSTATUS: {'SELESAI PENUH' if is_complete else f'BERHENTI ({hit_cutoff or "belum selesai"})'}")
print(f"Progress: {n_total_done} / {len(train_series)} series ({n_total_done/len(train_series)*100:.1f}%)")
print(f"Output: {running_bytes/1e9:.2f} GB")
print(f"{'='*60}")
if not is_complete:
    print(">>> UNTUK LANJUTKAN: tarik series_manifest.csv, jadikan 'prior_series_manifest.csv', push ulang.")


## 6. Ringkasan akhir

In [ ]:
final_manifest = pd.read_csv(MANIFEST_PATH)
print("Status distribusi:")
print(final_manifest["status"].value_counts())
print()
print("Laterality distribusi (dari series yang berhasil):")
print(final_manifest["Laterality"].value_counts(dropna=False).head(10))
print()
print(f"Total ukuran cache: {final_manifest['bytes'].fillna(0).sum()/1e9:.2f} GB")
print(f"Total series ter-cache OK: {(final_manifest['status']=='OK').sum()} / {len(train_series)}")


## Catatan

- **Reusable di notebook lain**: attach output ini sebagai Kaggle Dataset (`dataset_sources`), lalu baca `study_manifest.csv` + `series_manifest.csv` untuk join label/metadata, dan `np.load(cache_path)['slices']` untuk array (N,224,224) uint8 siap pakai.
- Kalau status bukan "SELESAI PENUH": tarik `series_manifest.csv`, taruh sebagai `prior_series_manifest.csv` di folder kernel ini, push ulang -- series yang sudah selesai otomatis di-skip (pola sama seperti audit exhaustive).